# Seminar HCI and BCI in practice
## Session 2 Simple preprocessing and artifact rejection
---

In [ ]:
# Environment Setting
import numpy as np
import os
import pickle
import matplotlib.pyplot as plt
import sys

sys.path.append(os.path.join(os.getcwd(), "src"))
from plot_std import plot_std
from remove_bad_epochs import remove_bad_epochs
from multitaper_spectrum import multitaper_spectrum

main_path = os.getcwd()
data_path = os.path.join(main_path, 'data/raw')
print(f'Now you are located: {main_path}')
print(f'Reading and Loading data are located: {data_path}')

---

## Load Baseline Corrected Data

The first preprocessing step is baseline correction, which was done in Session 1

Load data file to workspace (results from session 1 - already baseline corrected)

In [ ]:
# Load prepocessed data from Session 1, and then check the keys from ecog dict
ecog_file = os.path.join(data_path, 'ecogStruct1.pkl')
with open(ecog_file, 'rb') as f:
    ecog = pickle.load(f)

# Check data
print(ecog.keys())

---

## Rejecting bad channels

In the next step bad channels are rejected by visualizing their frequency content. The fourier transform of each channel is computed and compared with the mean and the standard deviation of all channels. We need set some parameters at first in the next code cells, after that we will excute a programm `removeBadChannels_periodogram_V2.py` to find out the bad channels.

In [ ]:
# Set sepctrum analysis parameters
params = {
    'tapers': [3, 5],  # TW(time-bandwidth product)=3, K(number of tapers)=5
    'pad': 0,  # no padding
    'Fs': 1000 / ecog['sampDur'],
    'fpass': [0, 200],
    'err': 0,
    'trialavg': False
}

# Multitaper strectral analysis
ecog['data'] = np.array(ecog['data']) # As data originally saved in a list
f, S = multitaper_spectrum(ecog, params)

# Create a new Dict to store multitaper spectral analysis results
periodogram = {
    'trailList': 1,
    'params': params,
    'periodogram':S,
    'centerFrequency':f
}

# update ecog dict
ecog['periodogram'] = periodogram

# Save the spectrual analysised data
with open(os.path.join(data_path, 'ecogStruct1_processed.pkl'), "wb") as file:
    pickle.dump(ecog, file)

# Check again your data
print(ecog.keys())
print(ecog['periodogram'].keys())
for key, value in ecog['periodogram'].items():
    print(f"Key: {key}, Type: {type(value).__name__}")

In [ ]:
ecog_file = os.path.join(data_path, 'ecogStruct1_periodogram.pkl')
with open(ecog_file, 'rb') as f:
    ecog = pickle.load(f)

print(ecog.keys())
print(ecog['periodogram'].keys())

<h2 style="color: #FF0000; font-weight: bold;">TASK 1:</h2>

Run the script `src/removeBadChannels_periodogram_V2.py` in **Anaconda Prompt** or **Bash** or **PowerShell** as you did in ```Session_01```. (Alternatively, you can use the `%run` magic command in another code cell: `%run src/removeBadChannels_periodogram_V2.py`. ***But this is not a general solution for all kinds of scripts, for example some functions with users active inputs/operations in terminal.***)

When running the function removeBadChannels a figure will appear (this may take a while!!), showing the mean plus/minus one standard deviation in black lines and a red line showing the frequency spectrum of the first channel. If the channel shows a spectrum considerably exceeding the standard deviation, it should be rejected.

If you succeed with running `removeBadChannels_periodogram_V2.py`, you will see the first channel plot, press ```Good``` to mark the channel as a good channel, press ```Bad``` for bad cahnnels, which later should be rejected. After you click the button, the second channel plot will show up. Do this for all channels. The data will then saved as `ecogStruct1_periodogram.pkl`.

In the end the function will store the indeces of the channels you rejected in the `dict` in  ```ecog['badChannels']```. 

In [ ]:
%run -i src/removeBadChannels_periodogram_V2.py

In [ ]:
# Reload the data after bad channels were marked
ecog_file = os.path.join(data_path, 'ecogStruct1_periodogram.pkl')
with open(ecog_file, 'rb') as f:
    ecog = pickle.load(f)
print(f'Channels {ecog['badChannels']} are marked as bad channels')

## Rejection of bad channels by variance estimation

An additional way of identifying bad channels is to look at the standard deviation and then deciding on a minimum and maximum standard deviation.

<h3 style="color: #FF0000; font-weight: bold;">TASK 1.1 (2 Point):</h3>

Calculate the standard deviation over the time series of each channel. 

<h3 style="color: #FF0000; font-weight: bold;">Fill in the missing parts (...) in the code below</h3>

In [ ]:
# standard deviation
ecogdata = np.array(ecog['data'])
s = ...

In [ ]:
plot_std(ecog, s)

<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration: </h3>
<div style="color: #FF0000; font-weight: bold;">What is a resonable std range(in muV) for a healthy channel? Write the range down, for next task. Why are these marked bad channels rejected?</div> 

---

<h2 style="color: #FF0000; font-weight: bold;">Task 2:</h2>

Choose the minimum and maximum standard deviation based on the plot.

<h3 style="color: #FF0000; font-weight: bold;">Define min and max threshold for std in the missing parts (...) in the code below</h3>

In [ ]:
# Choose the minimum and maximum standard deviation based on the plot.
minMaxStandardDeviation = [...]     # Set the min and max threshold for std

# Set bad channels and ensure there are no duplicates
idx = np.where((s < minMaxStandardDeviation[0]) | (s > minMaxStandardDeviation[1]))[0]
ecog['badChannels'] = np.unique(np.concatenate([ecog['badChannels'], idx]))

# Task 2.1 (1 pt):
# Save the numbers of the remaining 'good' channels in the ecog['selectedChannels']
# (hint: check out the documentation for numpy function setdiff1d: https://numpy.org/doc/2.0/reference/generated/numpy.setdiff1d.html)
# --------------------------------------------------------------------------
ecog['selectedChannels'] = ...
# ==========================================================================
ecog['selectedChannels']

---

<h2 style="color: #FF0000; font-weight: bold;">Task 3 (optional):</h2>

***Hint:*** *if you check the online documentation of numpy function `setdiff1d`, you will find the other relevant funtions to solve Task3*

<h3 style="color: #FF0000; font-weight: bold;">Fill in the missing parts (...) in the code below</h3>

In [ ]:
# Define arrays
a = np.array([1, 2, 3, 4, 5])
b = np.array([3, 4, 5, 6, 7])

# Task 3.1: What will c contain if c = setdiff(a, b)?
c = ...
print(f'3.1: Array c now is: {c}\n')

# Task 3.2: If you want c to contain [6, 7] instead, what should you change?
c = ...
print(f'3.2: Target array c is {[6,7]}, Array c now is: {c}\n')

# Task 3.3: What function can you use to get [3,4,5]?
c = ...
print(f'3.3: Target array c is {[3,4,5]}, Array c now is: {c}\n')

# Task 3.4: What function can you use to get [1,2,6,7]?
c = ...
print(f'3.4: Target array c is {[1,2,6,7]}, Array c now is: {c}\n')

# Task 3.5: What function can you use to get [1,2,3,4,5,6,7]?
c = ...
print(f'3.5: Target array c is {[1,2,3,4,5,6,7]}, Array c now is: {c}')


---
## Common Average Reference (CAR)

The last step is removing the common average reference (CAR). Here, the mean across all the selected (good) channels is removed from each channel(at each sampling point).

In [ ]:
# Selecting only the data from the good channels
data = ecog['data'][ecog['selectedChannels'] - 1, :]  # Adjust for 0-based indexing in Python

# Figure: Plot example data before re-referencing
fig, axes = plt.subplots(2, 1, figsize=(10, 5))

# First subplot (Before re-referencing)
axes[0].plot(data[14, 0:10000])  # Just a random channel and time points for demonstration
axes[0].set_title('Before Re-referencing')
axes[0].set_ylabel('Amplitude')

# DO NOT Close the plot window before the next subplot shows up!

<h2 style="color: #FF0000; font-weight: bold;">TASK 4.1 (1 pt):</h2>

<h3 style="color: #FF0000; font-weight: bold;">Fill in the missing parts (...) in the code below</h3>

In [ ]:
# Finding the new reference channel of the time series (TS)
ecog['refChanTS'] = ...  # Mean across all channels for each time point
data = ... # Subtract this reference Channel from each channel

In [ ]:
# Figure: Plot example data after re-referencing
# Second subplot (After re-referencing)
axes[1].plot(data[14, 0:10000])  # Same channel as before, now with CAR removed
axes[1].set_title('After Re-referencing')
axes[1].set_xlabel('Time Points')
axes[1].set_ylabel('Amplitude')

plt.tight_layout()
plt.show()

In [ ]:
# Saving the re-referenced data
ecog['data'][ecog['selectedChannels'] - 1, :] = data  # Save back to ECoG structure

# Save data for the next step
with open(os.path.join(data_path, 'ecogStruct1_periodogram.pkl'), 'wb') as f:
    pickle.dump(ecog, f)

<h2 style="color: #FF0000; font-weight: bold;">TASK 4.2 (Discussion, 1pt):</h2>

Why do we remove the common average reference? What are the advantages of the CAR? Are there disadvantages? Are there alternatives?
s have no brain activity.


<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration:</h3>

---

## Visual artefact removal

Finally remove artifacts by visual inspection of the remaining time series. 

<h2 style="color: #FF0000; font-weight: bold;">TASK 5(1 Pt):</h2>

```ecog_gui_v3.py```

To visually inspect the remaining time series the ```ecog_gui_v3.py (which is a TSGUI(TimeSeries Graphical User Interface))``` can be used. Change the number of viewed channels to make things a bit clearer. Then go through all the time series. If you find a bad (noisy) interval, press and hold `SHIFT` key to select [mouse click] the start and the end of the interval in the time series. **When it turns blue, press 'b' to add the interval to the list of bad intervals.** 


<h4 style="color: #0000FF; font-weight: bold;">GUI Manual:</h4>

**Ch**(value input): user can define which channels should be ploted in the interface, default [1, 40]

**Start**(value input): user define the left edge of the plot, default 0

**\#**(value input): user define how many channels will show up in the interface, default 40

**Interval**(value input): user define how many seconds are currently ploted in the interface, default 5

**up/down button**(functional button): used with **\#**, if there are only 20 channels in the interface now, use buttons to scroll up/down

**<</>> button**(functional button): scroll right/left with whole **interval**, if **interval=5**, then go right/left 5 seconds

**</> button**(functional button): scroll right/left with $\frac{1}{3}$ **interval**, if **interval=5**, then go right/left $\frac{5}{3}$ seconds

**vertical scale**(value input/functional button): lower the text `Vertical scale` area needs a user input, to define vertical scaling factor, defaut 1. **\*2 \\2 Button** used to halve or duplicate the scaling factor

**Interaction behavior**: user press **`SHIFT` on keyboard**, then user is able to use **mouse left click** to mark a point, which is selected to be the start time. **Holding on the `SHIFT` button**, user can click the second timepoint. Then this area will be **highlighted in blue**, by **pressing `B` on keyboard**, this interval will be saved and user can continue to mark next interval. **`ESC` on keyboard**, clear unsaved points(before **'B'** pressed). **Click `X(close button)` on the interface** to exit. Data will then saved into `ecogStruct1_badEpochs.pkl`.


In [ ]:
%run ecog_gui_v3.py

In [ ]:
# Load the data with marked bad intervals
ecog_file = os.path.join(data_path, 'ecogStruct1_badEpochs.pkl')
with open(ecog_file, 'rb') as f:
    ecog = pickle.load(f)

# Check data 
print(ecog.keys())

In [ ]:
# Sort bad intervals in ascending order
# making sure the bad intervals are in ascending order (in case they were marked in a different order)
selectedIntervalsInGUIUnits = np.array(ecog['marked_intervals']) # Ensure it's a NumPy array
sorted_indices = np.argsort(selectedIntervalsInGUIUnits[:, 0])  # Sort by the first column
selectedIntervalsInGUIUnits = selectedIntervalsInGUIUnits[sorted_indices]

# Add bad intervals to ecog structure
ecog['badIntervals'] = selectedIntervalsInGUIUnits

ecog['badIntervals']

In [ ]:
## Removing 'bad' epochs from the epoch structure

# Load file with epoch information
# provides onsets and labels of gesture epochs (hand labeled) (see Session 1)
epoch_file = os.path.join(data_path, 'epoch.pkl')
with open(epoch_file, 'rb') as f:
    epoch = pickle.load(f)

print(epoch.keys())
print(f'Before moving bad epochs, number of epochs: {len(epoch['OnsetIdx'])}')

# remove epochs overlapping with bad intervals
epoch = remove_bad_epochs(epoch,selectedIntervalsInGUIUnits)
print(f'After moving bad epochs, number of epochs: {epoch['OnsetIdx'].shape}')

In [ ]:
# Save current results 
epoch_file = os.path.join(data_path, 'epoch2.pkl')
with open(epoch_file, 'wb') as f:
    pickle.dump(epoch, f)
ecog_file = os.path.join(data_path, 'ecogStruct2.pkl')
with open(ecog_file, 'wb') as f:
    pickle.dump(ecog, f)

---
<h2 style="color: #FF0000; font-weight: bold;">Additional Task:</h2>

If you have some time left, have a closer look at the `remove_bad_epochs` function. How is the data from the GUI transformed in order to compare it to the onset information stored in the epoch structure? 

---